# Meteor trail detectability: can a radiant give a specular echo?

For a TX/RX pair, a radiant and a time, this notebook asks whether a meteor trail from that radiant can give a **specular trail echo**. It is geometry only: it says *possible*, not how strong or how likely. It does not use echo power.

A trail reflects specularly where the line is tangent to the surface of constant TX to point to RX path length. That point is found for each trail. It counts as **possible** when it is inside the ionised height band (default 70 to 120 km) and above the horizon of both stations.

**How to use:** fill in the form in Step 1 (your transmitter, receiver, radiant, date and time), then choose **Run All**. Each later step reads those values. After you change the form, choose **Run All** again.

Not modelled yet: zenith attraction, Earth's rotation velocity, deceleration, how long the trail lasts, echo strength.

## Step 1: your inputs

Fill in the form: transmitter (illuminator), receiver, radiant and time. Latitude is positive north and longitude is positive east, so US longitudes are negative. Heights are metres above sea level. The **Shower** menu fills in an approximate radiant (you can edit the numbers afterwards, or type your own RA and Dec). The form keeps what you typed when you choose Run All again. The cell under the form holds the other settings (height band, tolerance, plot sizes) as named values you can edit.


In [ ]:
import meteor_form

meteor_form.show_form()

In [ ]:
from datetime import datetime, timezone

# ---- Stations, radiant and time: read from the form above (change the form, then Run All) ----
F = meteor_form.values()
TX_NAME, TX = F.tx_name, F.tx            # (latitude deg, longitude deg, height m above sea level)
RX_NAME, RX = F.rx_name, F.rx
FREQ_MHZ = F.freq_mhz
RADIANT_NAME, RA_DEG, DEC_DEG = F.radiant_name, F.ra_deg, F.dec_deg      # J2000, degrees
WHEN = F.when                           # UTC

# ---- Assumptions (named inputs, change freely) ----
BAND_KM = (70.0, 120.0)    # ionised height band of a trail
TOL_DEG = 0.5              # trail-direction tolerance, degrees (about 45 Hz here; 100 Hz would be about 1.1 deg)

# ---- Step 3 map size: baselines to each side of the TX/RX midpoint (1.2 = about 250 km here; try 6 for the whole footprint) ----
MAP_HALF_BASELINES = 6.0

# ---- Step 5 only: size of the fixed ground box for the rate index, km to each side of the TX/RX midpoint ----
RATE_BOX_KM = 1500.0

# ---- Step 6 only: side of the square box for the zoomed strength map, nautical miles, centred on the TX/RX midpoint ----
ZOOM_BOX_NM = 300.0

# ---- Step 6b only: hour-by-hour zoomed maps. Minutes between maps, and grid cells per side in each small map ----
HOURLY_STEP_MIN = 60
HOURLY_N = 121

# ---- Step 6c only (prototype): floor for the polarization term and cap for the trail-aspect term, dB ----
POL_FLOOR_DB = -30.0
ASPECT_CAP_DB = 10.0

# ---- Step 7 only: link margins, dB below the strongest trail of the day (trails weaker than this are not counted) ----
MARGINS_DB = (3.0, 6.0, 10.0)

# ---- Step 2 only: the one trail to examine, given by where it crosses this height over a point ----
TRAIL_ALT_KM = 100.0
TRAIL_LAT, TRAIL_LON = None, None   # None = over the TX/RX midpoint

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import meteor_core as mc
import meteor_detect as md

mc.configure(TX, RX, FREQ_MHZ, 20.0)
mid = mc.point_on_path(0.5, 0)
print(f"{TX_NAME} to {RX_NAME}: baseline {mc.baseline_km():.0f} km, {FREQ_MHZ} MHz")

## Step 2: one trail

The radiant gives every trail the same direction at one moment. Here one trail is picked (it passes over the point set in Step 1) and its specular point is worked out.

In [ ]:
vhat = md.trail_direction(RA_DEG, DEC_DEG, WHEN)
lat0 = mid[0] if TRAIL_LAT is None else TRAIL_LAT
lon0 = mid[1] if TRAIL_LON is None else TRAIL_LON
head, desc = md.heading_descent(vhat, lat0, lon0)
sun = md.sun_alt_deg(WHEN, mid[0], mid[1])
print(f"{RADIANT_NAME} (RA {RA_DEG:g}, Dec {DEC_DEG:g}) at {WHEN:%Y-%m-%d %H:%M} UTC")
print(f"  radiant elevation over the TX/RX midpoint: {desc:5.1f} deg   (below 0 = radiant not up, no meteors from it)")
print(f"  trail heading {head:5.1f} deg from north, descent {desc:4.1f} deg below horizontal")
print(f"  sun elevation at the midpoint: {sun:5.1f} deg\n")

if desc <= 0:
    print("The radiant is below the horizon here, so nothing arrives from it at this time.")
else:
    r = md.trail_through(lat0, lon0, TRAIL_ALT_KM * 1e3, vhat, TX, RX, band_km=BAND_KM, tol_deg=TOL_DEG)
    print(f"Specular point of the trail through ({lat0:.3f}, {lon0:.3f}) at {TRAIL_ALT_KM:g} km:")
    print(f"  position            {r.lat:.3f} N, {r.lon:.3f} E, altitude {r.alt_km:.1f} km")
    print(f"  ranges              TX {r.range_tx_km:.0f} km, RX {r.range_rx_km:.0f} km")
    print(f"  scattering angle    {r.scatter_angle_deg:.1f} deg  (180 = straight back)")
    print(f"  point seen from the sites: TX {r.elev_tx_deg:.1f} deg, RX {r.elev_rx_deg:.1f} deg elevation")
    print(f"  altitude over the +-{TOL_DEG:g} deg tolerance cone: {r.alt_range_km[0]:.1f} to {r.alt_range_km[1]:.1f} km")
    print(f"  in height band {BAND_KM[0]:g}-{BAND_KM[1]:g} km: {r.in_band};  above both horizons: {r.visible}")
    print(f"\n  POSSIBLE (nominal direction): {r.possible};   possible within tolerance: {r.possible_within_tol}")

## Step 3: all trails from the radiant, on a map

Each point of the map is a trail from the radiant, labelled by where it crosses the height set in Step 2. **Green:** that trail has a specular point in the band. **Amber:** only if the trail direction is off by up to the tolerance. **Red:** no specular point in the band. Set MAP_HALF_BASELINES in Step 1 to zoom out: a low radiant can give a band of working trails more than a thousand kilometres long. The first run downloads map tiles (needs internet) and keeps them in `.tile_cache/`.

In [ ]:
if desc <= 0:
    print("Radiant below the horizon: nothing to draw.")
else:
    status = md.plot_radiant_map(vhat, TX, RX, WHEN,
        f"{RADIANT_NAME} {WHEN:%Y-%m-%d %H:%M} UTC: radiant {desc:.0f} deg up, trails heading {head:.0f} deg, sun {sun:.0f} deg",
        tol_deg=TOL_DEG, band_km=BAND_KM, half_baselines=MAP_HALF_BASELINES)
    plt.show()
    print(f"{100 * (status == 2).mean():.0f}% of the trail grid passes (nominal), {100 * (status >= 1).mean():.0f}% including the tolerance.")

## Step 4: through the day

The same test for every half hour of the UTC date in Step 1. How well a radiant works follows how low it is: shallow trails pass most, steep ones (radiant near overhead) pass a small share of this box. The dotted line is the radiant elevation. The dashed line is the sun elevation at the TX/RX midpoint, so you can see where those low-radiant times fall relative to sunrise and sunset. Grey shading marks times when the sun is more than 6 degrees below the horizon.

The percentages are the **share of the trail grid** (a box 1.2 baselines either side of the midpoint), so use them to compare times and radiants, not as a detection rate. The share depends on the size of the box: Step 5 uses a larger fixed box and weights by how much flux reaches the ground.

In [ ]:
rows = md.hourly_scan(RA_DEG, DEC_DEG, WHEN.date(), TX, RX, step_min=30, tol_deg=TOL_DEG, band_km=BAND_KM)

print(" UTC    radiant elev  heading  sun elev   pass(nominal)  pass(with tolerance)")
for q in rows:
    up = q["radiant_alt"] > 0
    print(f"{q['when']:%H:%M}   {q['radiant_alt']:8.1f}   {q['heading']:8.1f}  {q['sun_alt']:8.1f}   "
          + (f"{q['nominal']:9.1f}%  {q['with_tol']:14.1f}%" if up else "    radiant below horizon"))

hrs = [q["when"].hour + q["when"].minute / 60 for q in rows]
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(hrs, [q["nominal"] for q in rows], "g-", lw=2, label="pass, nominal")
ax.plot(hrs, [q["with_tol"] for q in rows], "-", color="orange", lw=1.5, label="pass, with tolerance")
ax.set_xlim(0, 24); ax.set_xlabel("UTC hour"); ax.set_ylabel("% of trail grid passing")
ax2 = ax.twinx()
ax2.plot(hrs, [q["sun_alt"] for q in rows], color="gold", lw=1.2, ls="--", label="sun elevation")
ax2.plot(hrs, [q["radiant_alt"] for q in rows], color="gray", lw=1.2, ls=":", label="radiant elevation")
ax2.axhline(0, color="k", lw=0.5); ax2.set_ylabel("elevation, deg")
ax.fill_between(hrs, 0, 1, where=[q["sun_alt"] < -6 for q in rows], transform=ax.get_xaxis_transform(), color="k", alpha=0.07)
h1, l1 = ax.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
ax.legend(h1 + h2, l1 + l2, loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=4, fontsize=8)
ax.set_title(f"{RADIANT_NAME}, {WHEN:%Y-%m-%d}: {TX_NAME} to {RX_NAME}")
plt.show()

## Step 5: a relative rate index (optional extra)

Step 4 gives the **share** of a box that passes. That is a share of an arbitrary box, and it ignores how much meteor flux reaches the ground. Meteors arrive uniformly over the ground, so the **area** of trails that pass (in a fixed box, set by RATE_BOX_KM in Step 1) is proportional to how many passing trails arrive per hour at one moment. Between different times a low radiant spreads the same flux over more ground, so the index is **pass area x sin(radiant elevation)**.

This is a **relative** index only. It has no meteor flux, brightness, antenna or echo-strength information, and it does not say what will be logged. Tested against the NN4SA Perseid log (6 m, half-hour bins), it correlates only weakly with echo counts (N4KH +0.39, AB4EJ +0.04), about the same as sin(radiant elevation) alone, so the data cannot yet confirm it. Sporadic meteors and the antenna beams are confounders. Treat it as the cheap, principled version of a rate measure, not a result.

**Box size matters.** For the default Geminid case the whole passing region fits inside the default 1500 km box when the radiant is above about 10 degrees, but for a lower radiant the passing region is larger than the box, so the index is only a lower bound there. Raise RATE_BOX_KM to check (the run takes longer). Also, a radiant near overhead passes only a small share of the small Step 4 box, but its passing trails are mostly farther out, so its index here can be the highest of the day.

In [ ]:
rate = md.rate_scan(RA_DEG, DEC_DEG, WHEN.date(), TX, RX, box_km=RATE_BOX_KM, tol_deg=TOL_DEG, band_km=BAND_KM)
best = max(q["index_km2"] for q in rate) or 1.0

print(f"Fixed box: +-{RATE_BOX_KM:g} km around the TX/RX midpoint ({(2 * RATE_BOX_KM) ** 2 / 1e6:.1f} million km2). Index shown as a % of the day's best.")
print(" UTC    radiant elev   pass area, thousand km2   index (area x sin elev), % of best")
for q in rate:
    print(f"{q['when']:%H:%M}   {q['radiant_alt']:8.1f}   "
          + (f"{q['area_km2'] / 1e3:16.0f}   {100 * q['index_km2'] / best:24.0f}%" if q["radiant_alt"] > 0 else "     radiant below horizon"))

hrs = [q["when"].hour + q["when"].minute / 60 for q in rate]
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(hrs, [q["area_km2"] / 1e3 for q in rate], "g-", lw=2, label="pass area")
ax.plot(hrs, [q["index_km2"] / 1e3 for q in rate], "-", color="tab:purple", lw=2, label="rate index: area x sin(radiant elevation)")
ax.set_xlim(0, 24); ax.set_ylim(bottom=0); ax.set_xlabel("UTC hour"); ax.set_ylabel("thousand km2 (relative, not a count)")
ax2 = ax.twinx()
ax2.plot(hrs, [q["sun_alt"] for q in rate], color="gold", lw=1.2, ls="--", label="sun elevation")
ax2.plot(hrs, [q["radiant_alt"] for q in rate], color="gray", lw=1.2, ls=":", label="radiant elevation")
ax2.axhline(0, color="k", lw=0.5); ax2.set_ylabel("elevation, deg")
ax.fill_between(hrs, 0, 1, where=[q["sun_alt"] < -6 for q in rate], transform=ax.get_xaxis_transform(), color="k", alpha=0.07)
h1, l1 = ax.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
ax.legend(h1 + h2, l1 + l2, loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=4, fontsize=8)
ax.set_title(f"{RADIANT_NAME}, {WHEN:%Y-%m-%d}: relative rate index, {TX_NAME} to {RX_NAME}")
plt.show()

## Step 6: relative strength of the passing trails (optional extra, prototype)

Not all passing trails are equally good. This step gives each passing trail a **relative strength in dB** from the bistatic range equation for an underdense trail: power goes roughly as 1 / (R_TX x R_RX x (R_TX + R_RX)), where R is the range from each station to the specular point. The strongest passing trail is set to 0 dB. Left: the same trail grid as Step 3, coloured by strength. Right: strength against how far the specular point lies from the TX/RX midpoint, measured across the trail direction (the way the band of working trails runs). The line is the strongest trail at each offset and the shading is the range of strengths at that offset.

**Read it as relative only.** It assumes isotropic antennas, so it ignores the antenna pattern, polarization and the scattering-angle term. It does not say whether any trail clears the noise floor. The peak sits near the stations because the range term rises steeply there, but a real horizontally polarised antenna sees almost nothing at that high an angle, so do not read the peak as the best place to be. The first test of weighting the Step 5 index by this strength did **not** improve the match with the NN4SA echo counts (N4KH correlation +0.39 fell to +0.08). Treat the picture as a guide to how quickly the geometry weakens away from the baseline, not a forecast.

In [ ]:
if desc <= 0:
    print("Radiant below the horizon: nothing to draw.")
else:
    lat_r, lon_r = md.grid_around(TX, RX, MAP_HALF_BASELINES)
    lat, lon, ok, db = md.strength_map(vhat, TX, RX, lat_r, lon_r, n=161, band_km=BAND_KM)
    if not ok.any():
        print("No trail from this radiant passes at this time, so there is nothing to compare.")
    else:
        pr = md.strength_profile(vhat, TX, RX, half_baselines=MAP_HALF_BASELINES, band_km=BAND_KM)
        fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 5.2), gridspec_kw={"width_ratios": [1, 1.2]})

        im = a1.pcolormesh(lon, lat, db - np.nanmax(db), cmap="viridis", shading="auto")
        a1.plot([TX[1], RX[1]], [TX[0], RX[0]], "k--", lw=1)
        a1.plot(TX[1], TX[0], "r^", ms=11, mec="k", label="TX"); a1.plot(RX[1], RX[0], "b^", ms=11, mec="k", label="RX")
        a1.set_xlim(lon_r); a1.set_ylim(lat_r); a1.set_aspect(1 / np.cos(np.radians(np.mean(lat_r))))
        a1.set_xlabel("longitude, deg"); a1.set_ylabel("latitude, deg"); a1.legend(loc="lower left", fontsize=8)
        a1.set_title("passing trails, relative strength (not passing = blank)")
        fig.colorbar(im, ax=a1, label="relative strength, dB (strongest = 0)", shrink=0.8)

        a2.fill_between(pr["offset_km"], pr["worst_db"], pr["best_db"], color="tab:blue", alpha=0.25, label="range at that offset")
        a2.plot(pr["offset_km"], pr["best_db"], color="tab:blue", lw=2, label="strongest trail at that offset")
        a2.axvline(pr["tx_offset_km"], color="r", ls=":", label="TX"); a2.axvline(pr["rx_offset_km"], color="b", ls=":", label="RX")
        for x in (-3, -6, -10):
            a2.axhline(x, color="gray", lw=0.6, ls="--")
        a2.set_xlabel("distance of the specular point from the TX/RX midpoint, across the trail direction, km")
        a2.set_ylabel("relative strength, dB"); a2.legend(fontsize=8, loc="lower center")
        a2.set_title("how fast the geometry weakens away from the baseline")
        plt.tight_layout(); plt.show()

        off, best_db = pr["offset_km"], pr["best_db"]
        print("Offsets (km, either side of the midpoint) where the strongest trail is within:")
        for x in (3, 6, 10, 20):
            sel = off[best_db >= -x]
            print(f"  {x:2d} dB of the best: {sel.min():6.0f} to {sel.max():5.0f} km" if sel.size else f"  {x:2d} dB of the best: none")
        print(f"All passing trails span {-pr['spread_db'].min():.0f} dB.")

**Zoomed in.** The same strength colours on a square box ZOOM_BOX_NM nautical miles on a side (set in Step 1; 300 nm is about 556 km), centred on the TX/RX midpoint and drawn over the map. The colour scale is the same as the wide map above, so a colour means the same number of dB in both. As in Step 3, each cell is a trail labelled by where it crosses 100 km altitude, not by where its echo happens. The lines mark 3, 6 and 10 dB below the strongest trail.

In [ ]:
if desc <= 0 or not ok.any():
    print("Nothing to draw: the radiant is below the horizon or no trail passes.")
else:
    half_km = ZOOM_BOX_NM * 1.852 / 2
    zlat, zlon = md.grid_around(TX, RX, half_km / mc.baseline_km())
    zl, zo, zok, zdb = md.strength_map(vhat, TX, RX, zlat, zlon, n=241, band_km=BAND_KM)
    if not zok.any():
        print("No trail passes inside this box.")
    else:
        best_wide = max(np.nanmax(db), np.nanmax(zdb))   # zero and colour scale shared with the wide map above
        rel = zdb - best_wide
        fig, ax = plt.subplots(figsize=(8.5, 7.5))
        img = md.background_image(zlat, zlon)
        if img is not None:
            ax.imshow(img, origin="lower", extent=(zlon[0], zlon[1], zlat[0], zlat[1]), aspect="auto", zorder=0)
        im = ax.pcolormesh(zo, zl, rel, cmap="viridis", vmin=np.nanmin(db) - best_wide, vmax=0, alpha=0.7, shading="auto", zorder=1)
        cs = ax.contour(zo, zl, np.ma.masked_invalid(rel), levels=[-10, -6, -3], colors=["white", "yellow", "red"], linewidths=1.3, zorder=2)
        ax.clabel(cs, fmt="%d dB", fontsize=8)
        ax.plot([TX[1], RX[1]], [TX[0], RX[0]], "k--", lw=1, zorder=3)
        ax.plot(TX[1], TX[0], "r^", ms=12, mec="k", zorder=4, label="TX"); ax.plot(RX[1], RX[0], "b^", ms=12, mec="k", zorder=4, label="RX")
        ax.set_xlim(zlon); ax.set_ylim(zlat); ax.set_aspect(1 / np.cos(np.radians(np.mean(zlat))))
        ax.set_xlabel("longitude, deg"); ax.set_ylabel("latitude, deg"); ax.legend(loc="lower left", fontsize=8, framealpha=0.9)
        ax.set_title(f"{RADIANT_NAME} {WHEN:%Y-%m-%d %H:%M} UTC: relative strength, {ZOOM_BOX_NM:g} nm ({2 * half_km:.0f} km) box")
        fig.colorbar(im, ax=ax, label="relative strength, dB (strongest trail = 0)", shrink=0.8)
        plt.tight_layout(); plt.show()
        print(f"{100 * zok.mean():.0f}% of the trails in this box pass. Strongest in the box: {np.nanmax(rel):.1f} dB, weakest passing: {np.nanmin(rel):.1f} dB.")

**Step 6b: hour by hour.** The zoomed map above for every HOURLY_STEP_MIN minutes of the day (set in Step 1; 60 is hourly), with the radiant above the horizon. One colour scale and one zero for every panel: 0 dB is the strongest trail of the whole day, so a colour means the same number of dB in every panel and you can see the strong areas appear, move and fade as the radiant moves. Each panel title gives the time (UTC), the radiant elevation and the share of the trails in the box that pass. The lines mark 3, 6 and 10 dB below the day's strongest trail. A panel with no colour means no trail in the box passes at that hour. Near a low radiant the passing trails can run past the edge of the box, so the box can understate them, and a radiant under 5 degrees (marked *) makes trails nearly level, so almost every cell passes and the picture is not reliable. Like everything in Steps 6 and 7 this is relative, from ranges only: no antennas or polarization.

In [ ]:
from datetime import timedelta

half_km = ZOOM_BOX_NM * 1.852 / 2
zlat, zlon = md.grid_around(TX, RX, half_km / mc.baseline_km())
wlat, wlon = md.grid_around(TX, RX, RATE_BOX_KM / mc.baseline_km())    # wide box, only used to find the day's strongest trail
day0 = datetime(WHEN.year, WHEN.month, WHEN.day, tzinfo=timezone.utc)

frames = []
for k in range(0, 24 * 60, HOURLY_STEP_MIN):
    t = day0 + timedelta(minutes=k)
    v = md.trail_direction(RA_DEG, DEC_DEG, t)
    el = md.heading_descent(v, mid[0], mid[1])[1]
    if el <= 0:
        continue
    _, _, wok, wdb = md.strength_map(v, TX, RX, wlat, wlon, n=121, band_km=BAND_KM)
    zl, zo, zok, zdb = md.strength_map(v, TX, RX, zlat, zlon, n=HOURLY_N, band_km=BAND_KM)
    tops = [np.nanmax(x[o]) for x, o in ((wdb, wok), (zdb, zok)) if o.any()]
    frames.append(dict(t=t, el=el, zl=zl, zo=zo, zok=zok, zdb=zdb, top=max(tops) if tops else np.nan))

best_day = np.nanmax([f["top"] for f in frames]) if frames else np.nan
if not frames or np.isnan(best_day):
    print("The radiant is never above the horizon, or no trail passes, on this day.")
else:
    low = min(np.nanmin(f["zdb"]) for f in frames if f["zok"].any()) - best_day if any(f["zok"].any() for f in frames) else -20.0
    ncols = 4
    nrows = -(-len(frames) // ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.2 * ncols, 3.9 * nrows), squeeze=False)
    img = md.background_image(zlat, zlon)
    last = None
    for ax, f in zip(axes.flat, frames):
        if img is not None:
            ax.imshow(img, origin="lower", extent=(zlon[0], zlon[1], zlat[0], zlat[1]), aspect="auto", zorder=0)
        rel = f["zdb"] - best_day
        if f["zok"].any():
            last = ax.pcolormesh(f["zo"], f["zl"], rel, cmap="viridis", vmin=low, vmax=0, alpha=0.75, shading="auto", zorder=1)
            ax.contour(f["zo"], f["zl"], np.ma.masked_invalid(rel), levels=[-10, -6, -3],
                       colors=["white", "yellow", "red"], linewidths=0.9, zorder=2)
        ax.plot([TX[1], RX[1]], [TX[0], RX[0]], "k--", lw=0.8, zorder=3)
        ax.plot(TX[1], TX[0], "r^", ms=8, mec="k", zorder=4); ax.plot(RX[1], RX[0], "b^", ms=8, mec="k", zorder=4)
        ax.set_xlim(zlon); ax.set_ylim(zlat); ax.set_aspect(1 / np.cos(np.radians(np.mean(zlat))))
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(f"{f['t']:%H:%M} UTC  el {f['el']:.0f} deg  {100 * f['zok'].mean():.0f}% pass" + (" *" if f["el"] < 5 else ""), fontsize=9)
    for ax in list(axes.flat)[len(frames):]:
        ax.axis("off")
    fig.suptitle(f"{RADIANT_NAME} {day0:%Y-%m-%d}: {TX_NAME} to {RX_NAME}, relative strength by hour, {ZOOM_BOX_NM:g} nm box "
                 f"(0 dB = strongest trail of the day)", fontsize=11)
    if any(f["el"] < 5 for f in frames):
        fig.text(0.01, 0.005, "* radiant under 5 deg: trails run nearly level, almost every cell passes and the picture is not reliable.", fontsize=8)
    fig.subplots_adjust(left=0.02, right=0.9, top=0.93, bottom=0.04, wspace=0.08, hspace=0.18)
    if last is not None:
        fig.colorbar(last, cax=fig.add_axes([0.925, 0.25, 0.012, 0.5]), label="relative strength, dB")
    plt.show()

    print("time UTC   radiant el   % pass   strongest in box (dB re the day's best)")
    for f in frames:
        s = f"{np.nanmax(f['zdb']) - best_day:6.1f}" if f["zok"].any() else "   none"
        print(f"  {f['t']:%H:%M}      {f['el']:5.1f}    {100 * f['zok'].mean():5.0f}    {s}")


**Step 6c: polarization and trail aspect (prototype, optional).** The strength above uses the ranges only. This step adds the two other geometry terms of the standard underdense-trail equation, still with isotropic antennas, so there are no antenna patterns. **Polarization:** with a horizontally polarized TX and RX and no Faraday rotation, the electrons in the trail are driven along the TX E-field, and the RX only accepts the horizontal part of what they re-radiate. The result depends only on the directions of the two rays at the specular point and can reach a null (floored at POL_FLOOR_DB here). **Aspect:** a trail scatters coherently over a length that grows as the trail lines up with the rays, so the echo is stronger the less square the trail is to the rays (capped at ASPECT_CAP_DB, since a real trail is not infinitely long). Left to right below: the range-only strength as in Step 6; the polarization term by itself, so the nulls are visible; and all three terms together. Range and total share one zero, the strongest range-only trail, so any dB lost or gained shows. These are relative dB from geometry alone: no noise floor, no antenna gain, and the H-pol assumption is yours to change in the code if your stations differ. This step needs meteor_polar.py and can be deleted without affecting anything else.

**How the two terms arise (one made-up trail, not your inputs).** The picture below shows a single specular trail on a flat Earth, with the transmitted wave going from the TX to the specular point P on the trail, and the scattered wave going from P to the RX. **Polarization:** a horizontally polarized TX sends an E-field that is horizontal and square to its ray (orange). The free electrons in the trail are driven along that direction and re-radiate, but a horizontally polarized RX only accepts the horizontal part of the wave arriving along its own ray (purple). Each arrow is fixed by the compass direction of its ray, so when the two rays point in different directions (seen from above, the middle panel) the arrows do not line up, and the received power falls as the cosine squared of the angle between them. It reaches a null when that angle is 90 degrees. The dark green arrows in the left panel are the scattered wave's own E-field: it is square to the scattered ray, so it is tilted out of the horizontal, and it keeps that tilt all the way to the RX, where the horizontal antenna accepts only its horizontal part (the dotted line). **Aspect:** the echo comes from the stretch of trail that adds in phase (the first Fresnel zone). That stretch is longer the more the trail lines up with the rays, so the echo is stronger by 1 over sine squared of the angle between the trail and the rays (right panel, capped in Step 1 by ASPECT_CAP_DB). The numbers in the picture use the same formulas as the maps below. To try other cases, change the arguments of `mp.explain_figure(...)`: `p_east_km`, `p_north_km` (where P sits relative to the middle of the baseline), `descent_deg` (how steeply the trail falls) and `aspect_demo_deg` (the lower sketch on the right).

In [ ]:
import meteor_polar as mp

fig, info = mp.explain_figure()          # try: mp.explain_figure(p_north_km=100) for a polarization null, or descent_deg=30
plt.show()
print(f"This example: the trail is {info['theta_deg']:.0f} deg from the rays (aspect {info['aspect_db']:+.1f} dB); "
      f"the two E arrows differ by {info['psi_deg']:.0f} deg (polarization {info['pol_db']:.1f} dB).")

In [ ]:
import meteor_polar as mp

half_km = ZOOM_BOX_NM * 1.852 / 2
zlat, zlon = md.grid_around(TX, RX, half_km / mc.baseline_km())
if md.heading_descent(vhat, mid[0], mid[1])[1] <= 0:
    print("Radiant below the horizon: nothing to draw.")
else:
    wlat, wlon = md.grid_around(TX, RX, MAP_HALF_BASELINES)
    wide = mp.strength_terms(vhat, TX, RX, wlat, wlon, n=161, band_km=BAND_KM, pol_floor_db=POL_FLOOR_DB, aspect_cap_db=ASPECT_CAP_DB)
    zoom = mp.strength_terms(vhat, TX, RX, zlat, zlon, n=241, band_km=BAND_KM, pol_floor_db=POL_FLOOR_DB, aspect_cap_db=ASPECT_CAP_DB)
    if not wide["ok"].any():
        print("No trail from this radiant passes at this time, so there is nothing to compare.")
    else:
        best = np.nanmax(wide["range_db"])                 # the zero for the range-only and the total maps
        total = zoom["range_db"] + zoom["pol_db"] + zoom["aspect_db"] - best
        rng = zoom["range_db"] - best
        vmin = min(np.nanmin(rng) if zoom["ok"].any() else 0, np.nanmin(total) if zoom["ok"].any() else 0)
        img = md.background_image(zlat, zlon)
        fig, axes = plt.subplots(1, 3, figsize=(17, 6))
        panels = [(rng, "range only (as Step 6)", "viridis", vmin, 0, "relative strength, dB (strongest range-only trail = 0)"),
                  (zoom["pol_db"], "polarization term alone (H-pol both ends)", "magma", POL_FLOOR_DB, 0, "dB lost to polarization"),
                  (total, "range + polarization + aspect", "viridis", vmin, 0, "relative strength, dB (same zero)")]
        for ax, (z, title, cmap, lo_, hi_, label) in zip(axes, panels):
            if img is not None:
                ax.imshow(img, origin="lower", extent=(zlon[0], zlon[1], zlat[0], zlat[1]), aspect="auto", zorder=0)
            im = ax.pcolormesh(zoom["lon"], zoom["lat"], z, cmap=cmap, vmin=lo_, vmax=hi_, alpha=0.75, shading="auto", zorder=1)
            if "polarization term" not in title and zoom["ok"].any():
                cs = ax.contour(zoom["lon"], zoom["lat"], np.ma.masked_invalid(z), levels=[-10, -6, -3], colors=["white", "yellow", "red"], linewidths=1.1, zorder=2)
                ax.clabel(cs, fmt="%d dB", fontsize=7)
            ax.plot([TX[1], RX[1]], [TX[0], RX[0]], "k--", lw=1, zorder=3)
            ax.plot(TX[1], TX[0], "r^", ms=10, mec="k", zorder=4); ax.plot(RX[1], RX[0], "b^", ms=10, mec="k", zorder=4)
            ax.set_xlim(zlon); ax.set_ylim(zlat); ax.set_aspect(1 / np.cos(np.radians(np.mean(zlat))))
            ax.set_xticks([]); ax.set_yticks([]); ax.set_title(title, fontsize=10)
            fig.colorbar(im, ax=ax, label=label, shrink=0.75)
        fig.suptitle(f"{RADIANT_NAME} {WHEN:%Y-%m-%d %H:%M} UTC, {ZOOM_BOX_NM:g} nm box: what polarization and trail aspect do to the strength", fontsize=11)
        plt.tight_layout(); plt.show()

        pr = mp.offset_profiles(vhat, TX, RX, half_baselines=MAP_HALF_BASELINES, band_km=BAND_KM, pol_floor_db=POL_FLOOR_DB, aspect_cap_db=ASPECT_CAP_DB)
        fig, ax = plt.subplots(figsize=(10, 4.8))
        ax.plot(pr["offset_km"], pr["range"], color="tab:blue", lw=2, label="range only")
        ax.plot(pr["offset_km"], pr["range_pol"], color="tab:orange", lw=2, label="range + polarization")
        ax.plot(pr["offset_km"], pr["range_pol_aspect"], color="tab:green", lw=2, ls="--", label="range + polarization + aspect")
        ax.axvline(pr["tx_offset_km"], color="r", ls=":", label="TX"); ax.axvline(pr["rx_offset_km"], color="b", ls=":", label="RX")
        for x in (-3, -6, -10):
            ax.axhline(x, color="gray", lw=0.6, ls="--")
        ax.set_xlabel("distance of the specular point from the TX/RX midpoint, across the trail direction, km")
        ax.set_ylabel("strongest trail at that offset, dB (range-only best = 0)")
        ax.set_title("how the extra terms change the strength profile of Step 6"); ax.legend(fontsize=8, loc="lower center")
        plt.tight_layout(); plt.show()

        ok = wide["ok"]
        r, p, a = wide["range_db"][ok], wide["pol_db"][ok], wide["aspect_db"][ok]
        near = r >= r.max() - 10
        print(f"Of {ok.sum()} passing trails in the Step 3 box, {100 * np.mean(p < -6):.0f}% lose more than 6 dB to polarization; among the {near.sum()} within 10 dB of the strongest range-only trail, "
              f"{100 * np.mean(p[near] < -6):.0f}% do (median loss {-np.median(p[near]):.1f} dB, worst {-p[near].min():.1f} dB).")
        print(f"Strongest trail: range only 0.0 dB, with polarization {np.max(r + p) - r.max():+.1f} dB, with polarization and aspect {np.max(r + p + a) - r.max():+.1f} dB (aspect adds up to {a.max():.1f} dB).")
        for name in ("range", "range_pol", "range_pol_aspect"):
            i = int(np.argmax(pr[name])); print(f"  {name:17s} strongest at {pr['offset_km'][i]:5.0f} km from the midpoint")


## Step 7: the rate index within a link margin (optional extra, prototype)

Step 5 counts every passing trail. Many real echoes are close to the noise floor, so a trail that is much weaker than the best may not be logged. This step repeats the Step 5 index, but counts only trails whose relative strength (Step 6) is within a chosen number of dB of the **strongest trail of the whole day**, so the cutoff means the same thing at every time step. Set the margins in MARGINS_DB in Step 1. The top plot is the Step 5 index with no limit; the lower plot is the same index limited to each margin (it has its own scale, since the limited values are much smaller).

**Read it as a planning picture, not a forecast.** Strength uses the range term only, with isotropic antennas, so it ignores the antenna pattern and polarization. A tight margin keeps only trails close to the baseline, which are at a high angle from the stations. A real low-takeoff antenna may favour the farther trails this step discounts. Tested against the NN4SA Perseid log, tight margins did **not** match the echo counts: for N4KH the correlation was +0.39 with no limit, +0.27 at 20 dB, and negative (-0.26 at 6 dB, -0.33 at 10 dB) with tighter limits. A likely reason, not tested, is that those echoes come when the radiant is high, when the passing trails are far from the stations. A tight margin favours low and middle radiant elevations and can drop to zero near an overhead radiant, but the logged echoes do not support that yet. As in Step 5, values for a radiant below about 10 degrees are lower bounds because the passing region is larger than the box. Relative only: it does not say what will be logged.

In [ ]:
marg = md.margin_scan(RA_DEG, DEC_DEG, WHEN.date(), TX, RX, margins_db=MARGINS_DB, box_km=RATE_BOX_KM, band_km=BAND_KM)

print(f"Rate index in thousand km2 (pass area x sin elevation), fixed +-{RATE_BOX_KM:g} km box; margins are dB below the day's strongest trail.")
print(" UTC    radiant elev   no limit" + "".join(f"   within {m:g} dB" for m in MARGINS_DB))
for q in marg:
    print(f"{q['when']:%H:%M}   {q['radiant_alt']:8.1f}   "
          + (f"{q['index_all_km2'] / 1e3:8.0f}" + "".join(f"{q['index_km2'][m] / 1e3:14.0f}" for m in MARGINS_DB)
             if q["radiant_alt"] > 0 else "   radiant below horizon"))

hrs = [q["when"].hour + q["when"].minute / 60 for q in marg]
dark = [q["sun_alt"] < -6 for q in marg]
fig, (top, ax) = plt.subplots(2, 1, figsize=(10, 7), sharex=True, gridspec_kw={"height_ratios": [1, 1.3]})
top.plot(hrs, [q["index_all_km2"] / 1e3 for q in marg], color="gray", lw=2)
top.set_ylim(bottom=0); top.set_ylabel("thousand km2"); top.set_title("no margin limit (the Step 5 index)")
for m, col in zip(MARGINS_DB, ("tab:red", "tab:orange", "tab:green", "tab:blue", "tab:purple")):
    ax.plot(hrs, [q["index_km2"][m] / 1e3 for q in marg], color=col, lw=2, label=f"within {m:g} dB of the day's best")
ax.set_xlim(0, 24); ax.set_ylim(bottom=0); ax.set_xlabel("UTC hour"); ax.set_ylabel("thousand km2 (relative, not a count)")
ax.set_title("only trails within a link margin of the day's strongest")
ax2 = ax.twinx()
ax2.plot(hrs, [q["sun_alt"] for q in marg], color="gold", lw=1.2, ls="--", label="sun elevation")
ax2.plot(hrs, [q["radiant_alt"] for q in marg], color="gray", lw=1.2, ls=":", label="radiant elevation")
ax2.axhline(0, color="k", lw=0.5); ax2.set_ylabel("elevation, deg")
for a in (top, ax):
    a.fill_between(hrs, 0, 1, where=dark, transform=a.get_xaxis_transform(), color="k", alpha=0.07)
h1, l1 = ax.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
ax.legend(h1 + h2, l1 + l2, loc="upper center", bbox_to_anchor=(0.5, -0.14), ncol=3, fontsize=8)
fig.suptitle(f"{RADIANT_NAME}, {WHEN:%Y-%m-%d}: rate index within a link margin, {TX_NAME} to {RX_NAME}")
plt.tight_layout(); plt.show()